# 02. Memory — 메시지로 대화 맥락 만들기

이 노트북에서는 [`docs/01.langchain.md`](../docs/01.langchain.md)의 **3장 Memory** 내용을 코드로 직접 실습합니다.

LLM은 기본적으로 **이전 대화를 기억하지 못합니다(stateless)**. 따라서 멀티턴 대화를 하려면, 이전 대화를 **메시지 리스트** 형태로 모델에 함께 전달해야 합니다.

다루는 주제는 다음과 같습니다.

1. **`SystemMessage` / `HumanMessage`** — 메시지 클래스로 대화 구성
2. **여러 메시지를 누적** 해서 모델에 전달하면 어떻게 되는지 확인
3. **딕셔너리 포맷** (OpenAI 스타일) 으로 동일하게 구성

| 메시지 종류         | 역할                                       | 딕셔너리의 role |
| :------------------ | :----------------------------------------- | :-------------- |
| **SystemMessage**   | 개발자가 사전에 작성한 입력 (프롬프트, 페르소나) | `system`        |
| **HumanMessage**    | 사용자의 요청(request), 입력                 | `user` / `human` |
| **AIMessage**       | Model의 답변 (result)                       | `assistant` / `ai` |

## 0. 환경 준비 및 모델 선언

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# .env 파일에서 환경 변수(예: OPENAI_API_KEY)를 로드
load_dotenv()

# 모델 선언 — 이번 노트북에서는 메시지 리스트를 다양한 방식으로 invoke해 본다.
model = init_chat_model("gpt-4o-mini")

## 1. `SystemMessage` + `HumanMessage` 로 대화 구성

가장 기본적인 대화 구성 방식입니다.

- **`SystemMessage`** : 모델에게 **역할(페르소나)** 이나 **규칙**을 주입
- **`HumanMessage`** : 실제 사용자의 질문/요청

이 두 가지를 **리스트**에 담아 `model.invoke(messages)` 에 전달하면, 모델은 SystemMessage의 페르소나를 유지한 채로 답변합니다.

In [ ]:
from langchain.messages import HumanMessage, SystemMessage

# SystemMessage로 페르소나를 지정하고, HumanMessage로 사용자 질문을 전달한다.
messages = [
  SystemMessage(content="You are a value investment investor."),  # 역할 부여: 가치 투자자
  HumanMessage(content="Broadcom price will be increased?")        # 사용자 질문
]

# invoke의 결과는 AIMessage 객체이다.
# response.content 에 답변 텍스트가, response.response_metadata 에 토큰 사용량 등이 담긴다.
response = model.invoke(messages)
print(response)

## 2. 메시지 리스트 = 메모리

LLM에는 진짜 "메모리"가 따로 있는 것이 아닙니다. 대신 **이전 대화를 메시지 리스트로 함께 보내주면**, 모델은 그것을 모두 읽고 마치 기억하는 것처럼 답변합니다.

아래 예시에서는

1. 첫 질문 ("Broadcom 가격이 오를까?")
2. "내가 좋아하는 색은 파란색이야"
3. "내가 어떤 색을 좋아한다고 했지?"
4. "내가 처음에 한 질문이 뭐였지?"

를 **하나의 `messages` 리스트** 에 모두 담아 한 번에 invoke합니다. 모델은 마지막 질문 ("처음에 한 질문이 뭐였지?")에 답변하기 위해 리스트 앞쪽 메시지를 참조합니다.

> 💡 즉, **메모리란 결국 메시지를 어떻게 누적·관리할 것인가**의 문제입니다.

In [ ]:
# 여러 개의 HumanMessage를 누적해 보낸다.
# 모델은 리스트 전체를 "이전 대화"로 보고 마지막 질문에 답한다.
messages = [
  SystemMessage(content="You are a value investment investor."),
  HumanMessage(content="Broadcom price will be increased?"),

  HumanMessage(content="My favorite color is blue."),
  HumanMessage(content="What color did I say is my favorite?"),
  HumanMessage(content="Remind me what my first question was.")
]

# 마지막 질문은 "내가 처음에 한 질문이 뭐였지?" → 모델이 messages[1]을 참조해 답변한다.
response = model.invoke(messages)
print(response)

## 3. 딕셔너리 포맷 (OpenAI 스타일)

LangChain의 메시지 클래스 대신, **OpenAI API와 동일한 딕셔너리 형식** 으로도 메시지를 전달할 수 있습니다.

#### Role 매핑

| LangChain 클래스   | 딕셔너리 role   |
| :----------------- | :-------------- |
| `SystemMessage`    | `"system"`      |
| `HumanMessage`     | `"user"` 또는 `"human"` |
| `AIMessage`        | `"assistant"` 또는 `"ai"` |

> 💡 어떤 표기를 쓰더라도 LangChain 내부에서 동일한 메시지 객체로 변환되므로 결과는 같아야 합니다. 기존 OpenAI 코드를 LangChain으로 옮길 때 변환 부담이 적은 게 장점입니다.

In [ ]:
# 메시지를 딕셔너리(role + content) 형태로 정의 — OpenAI API와 동일한 포맷
# 위 셀의 SystemMessage / HumanMessage 예시와 동일한 의미를 갖는다.
messages_data = [
  {"role": "system", "content": "You are a value investment investor."},
  {"role": "human",  "content": "Broadcom price will be increased?"},
  {"role": "human",  "content": "My favorite color is blue."},
  {"role": "human",  "content": "What color did I say is my favorite?"},
  {"role": "human",  "content": "Remind me what my first question was."}
]

# 결과는 위 셀과 동일하게 첫 질문(Broadcom 관련)을 회상하는 답변이 나온다.
response = model.invoke(messages_data)
print(response)

## 📚 정리

- LLM은 **stateless** — 메모리는 메시지 리스트를 누적·재전송해 구현한다.
- 메시지는 **객체 방식(`SystemMessage` / `HumanMessage` / `AIMessage`)** 또는 **딕셔너리 방식(`role` + `content`)** 으로 전달할 수 있다.
- 두 방식은 서로 호환되며, 같은 결과를 낸다.
- 실제 챗봇에서는 모델 응답(AIMessage)도 다시 `messages` 에 누적해 다음 invoke 때 함께 보내야 **멀티턴 대화** 가 된다.

> 다음 노트북부터는 LangGraph로 이 메시지 누적 작업을 자동화해 본격적인 Agent를 구축합니다.